In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.model_selection import GridSearchCV
from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV


from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix)

In [2]:
TW_500= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=500/Twitter-Relative-Sigma-500.data",
    sep=",",
    header=None
)

TW_1000= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=1000/Twitter-Relative-Sigma-1000.data",
    sep=",",
    header=None
)
TW_1500= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=1500/Twitter-Relative-Sigma-1500.data",
    sep=",",
    header=None
)
groups = ["NCD", 'AI', 'AS(NA)', 'BL',
         'NAC', 'AS(NAC)', 'CS', 'AT', 'NA','ADL', 'NAD']

columns = []
for group in groups:
    for t in range(7):
        columns.append(f"{group}_{t}")

columns.append("label") 

TW_500.columns = columns
TW_1000.columns = columns
TW_1500.columns = columns


### 500

### 1 Baseline Random Forest

In [5]:
X = TW_500.drop(columns=['label'])
y = TW_500['label']

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, stratify=y,random_state=42)
rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

#
accuracy_rf = accuracy_score(y_test, y_pred)
precision_rf = precision_score(y_test, y_pred)
recall_rf = recall_score(y_test, y_pred)
f1_rf = f1_score(y_test, y_pred)
roc_auc_rf = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print('1.Baseline Random Forest')
print("Accuracy:", accuracy_rf)
print("Precision:", precision_rf)
print("Recall:", recall_rf)
print("F1-score:", f1_rf)
print("ROC-AUC:", roc_auc_rf)


print("Confusion Matrix:")
print(cm)

1.Baseline Random Forest
Accuracy: 0.983974131191813
Precision: 0.8321167883211679
Recall: 0.4723756906077348
F1-score: 0.6026431718061674
ROC-AUC: 0.9634550930166859
Confusion Matrix:
[[27349    69]
 [  382   342]]


### 2.Random Forest with Stratified K-Fold

In [6]:
X = TW_500.drop(columns=['label'])
y = TW_500['label']

rf = RandomForestClassifier(random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}


cv_results = cross_validate( rf,X,y,cv=skf, scoring=scoring)

print('2.Random Forest with Stratified K-Fold')
print("Accuracy:", cv_results['test_accuracy'].mean())
print("Precision:", cv_results['test_precision'].mean())
print("Recall:", cv_results['test_recall'].mean())
print("F1-score:", cv_results['test_f1'].mean())
print("ROC-AUC:", cv_results['test_roc_auc'].mean())

2.Random Forest with Stratified K-Fold
Accuracy: 0.9825381812150354
Precision: 0.7783513107709263
Recall: 0.44944751381215475
F1-score: 0.5697769705118223
ROC-AUC: 0.9584993474059125


### 3.Random Forest with Grid Search

In [8]:
X = TW_500.drop(columns=['label'])
y = TW_500['label']

rf = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}


grid_search = GridSearchCV( estimator=rf,param_grid=param_grid,cv=5,scoring='f1',n_jobs=-1).fit(X_train, y_train)

best_rf = grid_search.best_estimator_

y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:,1]

print('3.Random Forest with Grid Search')
print("Best parameters:", grid_search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

3.Random Forest with Grid Search
Best parameters: {'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
Accuracy: 0.9838319948830929
Precision: 0.8210023866348448
Recall: 0.47513812154696133
F1-score: 0.6019247594050744
ROC-AUC: 0.9605202746189642


### 4.Balanced Random Forest

In [6]:
X = TW_500.drop(columns=['label'])
y = TW_500['label']


X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, stratify=y,random_state=42)
rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)


brf = BalancedRandomForestClassifier(n_estimators=200,random_state=42).fit(X_train, y_train)


y_pred = brf.predict(X_test)
y_prob = brf.predict_proba(X_test)[:,1]

accuracy_brf = accuracy_score(y_test, y_pred)
precision_brf = precision_score(y_test, y_pred)
recall_brf = recall_score(y_test, y_pred)
f1_brf = f1_score(y_test, y_pred)
roc_auc_brf = roc_auc_score(y_test, y_prob)

print("4. Balanced Random Forest")
print("Accuracy:", accuracy_brf)
print("Precision:", precision_brf)
print("Recall:", recall_brf)
print("F1-score:", f1_brf)
print("ROC-AUC:", roc_auc_brf)


cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

4. Balanced Random Forest
Accuracy: 0.913652192452562
Precision: 0.21375838926174498
Recall: 0.8798342541436464
F1-score: 0.3439524838012959
ROC-AUC: 0.9688054012587608
Confusion Matrix:
[[25075  2343]
 [   87   637]]


### Random Forest (SMOTE + Random Search)

In [5]:
X = TW_500.drop(columns=['label'])
y = TW_500['label']


X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, stratify=y,random_state=42)
pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('rf', RandomForestClassifier(random_state=42))
])

param_dist = {
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [10, 20, None],
    'rf__min_samples_split': [2, 5],
    'rf__min_samples_leaf': [1, 2],
    'rf__max_features': ['sqrt', 'log2']
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=15,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)

best_rf = search.best_estimator_

y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:, 1]

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("Random Forest (SMOTE + Random Search)")
print("Best params:", search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Random Forest (SMOTE + Random Search)
Best params: {'rf__n_estimators': 200, 'rf__min_samples_split': 2, 'rf__min_samples_leaf': 1, 'rf__max_features': 'sqrt', 'rf__max_depth': None}
Accuracy: 0.9675929216118258
Precision: 0.4195205479452055
Recall: 0.6767955801104972
F1: 0.5179704016913319
ROC-AUC: 0.9672814195537953


In [17]:
import pandas as pd

rf_500 = [
    {"Model": "RF baseline", "Accuracy": 0.98397, "Precision": 0.8321, "Recall": 0.4724, "F1": 0.6026, "ROC-AUC": 0.9635},
    {"Model": "RF CV", "Accuracy": 0.98254, "Precision": 0.7784, "Recall": 0.4494, "F1": 0.5698, "ROC-AUC": 0.9585},
    {"Model": "RF tuned", "Accuracy": 0.98383, "Precision": 0.8210, "Recall": 0.4751, "F1": 0.6019, "ROC-AUC": 0.9605},
    {"Model": "RF balanced", "Accuracy": 0.91365, "Precision": 0.2138, "Recall": 0.8798, "F1": 0.3440, "ROC-AUC": 0.9688}
]

df_rf_500 = pd.DataFrame(rf_500)
df_rf_500.sort_values(by="F1", ascending=False).round(4)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,RF baseline,0.9840,0.8321,0.4724,0.6026,0.9635
2,RF tuned,0.9838,0.8210,0.4751,0.6019,0.9605
1,RF CV,0.9825,0.7784,0.4494,0.5698,0.9585
3,RF balanced,0.9136,0.2138,0.8798,0.3440,0.9688


### 1000

### 1 Baseline Random Forest

In [6]:
X = TW_1000.drop(columns=['label'])
y = TW_1000['label']

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, stratify=y,random_state=42)
rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

#
accuracy_rf = accuracy_score(y_test, y_pred)
precision_rf = precision_score(y_test, y_pred)
recall_rf = recall_score(y_test, y_pred)
f1_rf = f1_score(y_test, y_pred)
roc_auc_rf = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print('1.Baseline Random Forest')
print("Accuracy:", accuracy_rf)
print("Precision:", precision_rf)
print("Recall:", recall_rf)
print("F1-score:", f1_rf)
print("ROC-AUC:", roc_auc_rf)


print("Confusion Matrix:")
print(cm)

1.Baseline Random Forest
Accuracy: 0.995416104043778
Precision: 0.83125
Recall: 0.5659574468085107
F1-score: 0.6734177215189874
ROC-AUC: 0.9559941111396592
Confusion Matrix:
[[27880    27]
 [  102   133]]


### 2.Random Forest with Stratified K-Fold


In [8]:
X = TW_1000.drop(columns=['label'])
y = TW_1000['label']

rf = RandomForestClassifier(random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}


cv_results = cross_validate( rf,X,y,cv=skf, scoring=scoring)

print('2.Random Forest with Stratified K-Fold')
print("Accuracy:", cv_results['test_accuracy'].mean())
print("Precision:", cv_results['test_precision'].mean())
print("Recall:", cv_results['test_recall'].mean())
print("F1-score:", cv_results['test_f1'].mean())
print("ROC-AUC:", cv_results['test_roc_auc'].mean())

2.Random Forest with Stratified K-Fold
Accuracy: 0.9955865744258375
Precision: 0.849704736604178
Recall: 0.5760043274432023
F1-score: 0.6856887654376701
ROC-AUC: 0.966988806233714


### 3.Random Forest with Grid Search

In [9]:
X = TW_1000.drop(columns=['label'])
y = TW_1000['label']

rf = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}


grid_search = GridSearchCV( estimator=rf,param_grid=param_grid,cv=5,scoring='f1',n_jobs=-1).fit(X_train, y_train)

best_rf = grid_search.best_estimator_

y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:,1]

print('3.Random Forest with Grid Search')
print("Best parameters:", grid_search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

3.Random Forest with Grid Search
Best parameters: {'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Accuracy: 0.9953450358894179
Precision: 0.8170731707317073
Recall: 0.5702127659574469
F1-score: 0.6716791979949874
ROC-AUC: 0.9616641596061082


### 4.Balanced Random Forest

In [10]:
X = TW_1000.drop(columns=['label'])
y = TW_1000['label']


X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, stratify=y,random_state=42)
rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)


brf = BalancedRandomForestClassifier(n_estimators=200,random_state=42).fit(X_train, y_train)


y_pred = brf.predict(X_test)
y_prob = brf.predict_proba(X_test)[:,1]

accuracy_brf = accuracy_score(y_test, y_pred)
precision_brf = precision_score(y_test, y_pred)
recall_brf = recall_score(y_test, y_pred)
f1_brf = f1_score(y_test, y_pred)
roc_auc_brf = roc_auc_score(y_test, y_prob)

print("4. Balanced Random Forest")
print("Accuracy:", accuracy_brf)
print("Precision:", precision_brf)
print("Recall:", recall_brf)
print("F1-score:", f1_brf)
print("ROC-AUC:", roc_auc_brf)


cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

4. Balanced Random Forest
Accuracy: 0.9539123018975197
Precision: 0.1426648721399731
Recall: 0.902127659574468
F1-score: 0.2463683904706566
ROC-AUC: 0.9841831798473502
Confusion Matrix:
[[26633  1274]
 [   23   212]]


### Random Forest (SMOTE + Random Search)

In [11]:
X = TW_1000.drop(columns=['label'])
y = TW_1000['label']


X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, stratify=y,random_state=42)
pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('rf', RandomForestClassifier(random_state=42))
])

param_dist = {
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [10, 20, None],
    'rf__min_samples_split': [2, 5],
    'rf__min_samples_leaf': [1, 2],
    'rf__max_features': ['sqrt', 'log2']
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=15,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)

best_rf = search.best_estimator_

y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:, 1]

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("Random Forest (SMOTE + Random Search)")
print("Best params:", search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Random Forest (SMOTE + Random Search)
Best params: {'rf__n_estimators': 200, 'rf__min_samples_split': 5, 'rf__min_samples_leaf': 1, 'rf__max_features': 'log2', 'rf__max_depth': None}
Accuracy: 0.9921825030203966
Precision: 0.5222551928783383
Recall: 0.7489361702127659
F1: 0.6153846153846154
ROC-AUC: 0.9802409065368333


In [19]:
import pandas as pd

rf_1000 = [
    {"Model": "RF baseline", "Accuracy": 0.99542, "Precision": 0.8313, "Recall": 0.5660, "F1": 0.6734, "ROC-AUC": 0.9560},
    {"Model": "RF CV", "Accuracy": 0.99559, "Precision": 0.8497, "Recall": 0.5760, "F1": 0.6857, "ROC-AUC": 0.9670},
    {"Model": "RF tuned", "Accuracy": 0.99535, "Precision": 0.8171, "Recall": 0.5702, "F1": 0.6717, "ROC-AUC": 0.9617},
    {"Model": "RF + SMOTE (Random Search)", "Accuracy": 0.99218, "Precision": 0.5223, "Recall": 0.7489, "F1": 0.6154, "ROC-AUC": 0.9802},
    {"Model": "RF balanced", "Accuracy": 0.95391, "Precision": 0.1427, "Recall": 0.9021, "F1": 0.2464, "ROC-AUC": 0.9842}
]

df_rf_1000 = pd.DataFrame(rf_1000)
df_rf_1000

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,RF baseline,0.99542,0.8313,0.5660,0.6734,0.9560
1,RF CV,0.99559,0.8497,0.5760,0.6857,0.9670
2,RF tuned,0.99535,0.8171,0.5702,0.6717,0.9617
3,RF + SMOTE (Random Search),0.99218,0.5223,0.7489,0.6154,0.9802
4,RF balanced,0.95391,0.1427,0.9021,0.2464,0.9842


### 1500

### 1 Baseline Random Forest

In [12]:
X = TW_1500.drop(columns=['label'])
y = TW_1500['label']

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, stratify=y,random_state=42)
rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

#
accuracy_rf = accuracy_score(y_test, y_pred)
precision_rf = precision_score(y_test, y_pred)
recall_rf = recall_score(y_test, y_pred)
f1_rf = f1_score(y_test, y_pred)
roc_auc_rf = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print('1.Baseline Random Forest')
print("Accuracy:", accuracy_rf)
print("Precision:", precision_rf)
print("Recall:", recall_rf)
print("F1-score:", f1_rf)
print("ROC-AUC:", roc_auc_rf)


print("Confusion Matrix:")
print(cm)

1.Baseline Random Forest
Accuracy: 0.997690284983299
Precision: 0.8
Recall: 0.4489795918367347
F1-score: 0.5751633986928104
ROC-AUC: 0.9776697478306684
Confusion Matrix:
[[28033    11]
 [   54    44]]


### 2.Random Forest with Stratified K-Fold

In [13]:
X = TW_1500.drop(columns=['label'])
y = TW_1500['label']

rf = RandomForestClassifier(random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}


cv_results = cross_validate( rf,X,y,cv=skf, scoring=scoring)

print('2.Random Forest with Stratified K-Fold')
print("Accuracy:", cv_results['test_accuracy'].mean())
print("Precision:", cv_results['test_precision'].mean())
print("Recall:", cv_results['test_recall'].mean())
print("F1-score:", cv_results['test_f1'].mean())
print("ROC-AUC:", cv_results['test_roc_auc'].mean())

2.Random Forest with Stratified K-Fold
Accuracy: 0.9980100484930488
Precision: 0.8327960372932456
Recall: 0.5348832316431728
F1-score: 0.6504011507021839
ROC-AUC: 0.9773922385573204


### 3.Random Forest with Grid Search

In [14]:
X = TW_1500.drop(columns=['label'])
y = TW_1500['label']

rf = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}


grid_search = GridSearchCV( estimator=rf,param_grid=param_grid,cv=5,scoring='f1',n_jobs=-1).fit(X_train, y_train)

best_rf = grid_search.best_estimator_

y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:,1]

print('3.Random Forest with Grid Search')
print("Best parameters:", grid_search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

3.Random Forest with Grid Search
Best parameters: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
Accuracy: 0.997619216828939
Precision: 0.7818181818181819
Recall: 0.4387755102040816
F1-score: 0.5620915032679739
ROC-AUC: 0.9723741336500369


### 4.Balanced Random Forest

In [15]:
X = TW_1500.drop(columns=['label'])
y = TW_1500['label']


X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, stratify=y,random_state=42)
rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)


brf = BalancedRandomForestClassifier(n_estimators=200,random_state=42).fit(X_train, y_train)


y_pred = brf.predict(X_test)
y_prob = brf.predict_proba(X_test)[:,1]

accuracy_brf = accuracy_score(y_test, y_pred)
precision_brf = precision_score(y_test, y_pred)
recall_brf = recall_score(y_test, y_pred)
f1_brf = f1_score(y_test, y_pred)
roc_auc_brf = roc_auc_score(y_test, y_prob)

print("4. Balanced Random Forest")
print("Accuracy:", accuracy_brf)
print("Precision:", precision_brf)
print("Recall:", recall_brf)
print("F1-score:", f1_brf)
print("ROC-AUC:", roc_auc_brf)


cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

4. Balanced Random Forest
Accuracy: 0.9775779972994101
Precision: 0.1313969571230982
Recall: 0.9693877551020408
F1-score: 0.23142509135200975
ROC-AUC: 0.9932971584012296
Confusion Matrix:
[[27416   628]
 [    3    95]]


### Random Forest (SMOTE + Random Search)

In [16]:
X = TW_1500.drop(columns=['label'])
y = TW_1500['label']


X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, stratify=y,random_state=42)
pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('rf', RandomForestClassifier(random_state=42))
])

param_dist = {
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [10, 20, None],
    'rf__min_samples_split': [2, 5],
    'rf__min_samples_leaf': [1, 2],
    'rf__max_features': ['sqrt', 'log2']
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=15,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)

best_rf = search.best_estimator_

y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:, 1]

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("Random Forest (SMOTE + Random Search)")
print("Best params:", search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Random Forest (SMOTE + Random Search)
Best params: {'rf__n_estimators': 200, 'rf__min_samples_split': 5, 'rf__min_samples_leaf': 1, 'rf__max_features': 'log2', 'rf__max_depth': None}
Accuracy: 0.9964821263591784
Precision: 0.4966442953020134
Recall: 0.7551020408163265
F1: 0.5991902834008097
ROC-AUC: 0.9857412477185995


In [20]:
import pandas as pd

rf_1500 = [
    {"Model": "RF baseline", "Accuracy": 0.99769, "Precision": 0.8000, "Recall": 0.4490, "F1": 0.5752, "ROC-AUC": 0.9777},
    {"Model": "RF CV", "Accuracy": 0.99801, "Precision": 0.8328, "Recall": 0.5349, "F1": 0.6504, "ROC-AUC": 0.9774},
    {"Model": "RF tuned", "Accuracy": 0.99762, "Precision": 0.7818, "Recall": 0.4388, "F1": 0.5621, "ROC-AUC": 0.9724},
    {"Model": "RF + SMOTE", "Accuracy": 0.99648, "Precision": 0.4966, "Recall": 0.7551, "F1": 0.5992, "ROC-AUC": 0.9857},
    {"Model": "RF balanced", "Accuracy": 0.97758, "Precision": 0.1314, "Recall": 0.9694, "F1": 0.2314, "ROC-AUC": 0.9933}
]

df_rf_1500 = pd.DataFrame(rf_1500)
df_rf_1500.sort_values(by="F1", ascending=False).round(4)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
1,RF CV,0.9980,0.8328,0.5349,0.6504,0.9774
3,RF + SMOTE,0.9965,0.4966,0.7551,0.5992,0.9857
0,RF baseline,0.9977,0.8000,0.4490,0.5752,0.9777
2,RF tuned,0.9976,0.7818,0.4388,0.5621,0.9724
4,RF balanced,0.9776,0.1314,0.9694,0.2314,0.9933
